# OpenET vs NLDAS Noah ET Harmonization — Iowa 2019–2023

Computes the monthly delta between OpenET (actual ET including irrigation)
and NLDAS Noah ET (modeled ET without irrigation) to isolate the human
contribution to evapotranspiration in Iowa.

**Delta = OpenET − NLDAS_Noah_ET**
- Positive values → more ET than the land surface model predicts → likely irrigation
- Negative values → less ET than expected → drought, crop stress, or fallow land

**Data sources:**
- OpenET: `data/raw/OpenET/OpenET_Iowa_YYYYMM.tif` (mm/month, 0.125°)
- NLDAS Noah: `data/raw/NLDAS_Noah/NLDAS_NOAH0125_M.AYYYYMM.020.nc` — variable `EVPsfc` (mm/month, 0.125°)

**Output:** `data/processed/ET_delta/ET_delta_YYYYMM_Iowa.tif`

In [ ]:
import xarray as xr
import rioxarray as rxr
import rasterio
import geopandas as gpd
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded successfully!')

## 1. Paths and Configuration

In [ ]:
project_root = Path('../../..').resolve()

nldas_dir  = project_root / 'data' / 'raw' / 'NLDAS_Noah'
openet_dir = project_root / 'data' / 'raw' / 'OpenET'
output_dir = project_root / 'data' / 'processed' / 'ET_delta'
figures_dir = project_root / 'figures' / 'et_delta'
iowa_boundary = project_root / 'data' / 'aoi' / 'iowa.geojson'

output_dir.mkdir(parents=True, exist_ok=True)
figures_dir.mkdir(parents=True, exist_ok=True)

YEARS = list(range(2019, 2024))
MONTHS = list(range(1, 13))

# Verify input files exist
nldas_files  = sorted(nldas_dir.glob('NLDAS_NOAH0125_M.A*.nc'))
openet_files = sorted(openet_dir.glob('OpenET_Iowa_*.tif'))

print(f'NLDAS Noah files found:  {len(nldas_files)} (expected 60)')
print(f'OpenET files found:      {len(openet_files)} (expected 60)')
print(f'Output directory:        {output_dir}')

## 2. Verify Spatial Alignment

Both datasets should be on the NLDAS 0.125° grid. Check one pair to confirm.

In [ ]:
iowa_gdf = gpd.read_file(iowa_boundary)

# Sample one NLDAS and one OpenET file
sample_nldas  = nldas_files[0]
sample_openet = openet_files[0]

# NLDAS grid
ds_nldas = xr.open_dataset(sample_nldas)
print('NLDAS Noah grid:')
print(f'  lon: {float(ds_nldas.lon[0]):.4f} to {float(ds_nldas.lon[-1]):.4f}, step={float(ds_nldas.lon[1]-ds_nldas.lon[0]):.4f}')
print(f'  lat: {float(ds_nldas.lat[0]):.4f} to {float(ds_nldas.lat[-1]):.4f}, step={float(ds_nldas.lat[1]-ds_nldas.lat[0]):.4f}')
ds_nldas.close()

# OpenET grid
with rasterio.open(sample_openet) as src:
    t = src.transform
    print('\nOpenET grid:')
    print(f'  xOrigin={t.c:.4f}, yOrigin={t.f:.4f}')
    print(f'  xScale={t.a:.4f}, yScale={t.e:.4f}')
    print(f'  CRS: {src.crs}')
    print(f'  Shape: {src.height} rows x {src.width} cols')

## 3. Process All 60 Months — Compute Delta

In [ ]:
records = []
delta_grids = {}  # keyed by (year, month)
skipped = []

for year in YEARS:
    for month in MONTHS:
        yyyymm = f'{year}{month:02d}'

        nldas_path  = nldas_dir  / f'NLDAS_NOAH0125_M.A{yyyymm}.020.nc'
        openet_path = openet_dir / f'OpenET_Iowa_{yyyymm}.tif'
        out_path    = output_dir / f'ET_delta_{yyyymm}_Iowa.tif'

        if not nldas_path.exists() or not openet_path.exists():
            skipped.append(yyyymm)
            print(f'  SKIP {yyyymm}: missing file(s)')
            continue

        # ── Load NLDAS Noah EVPsfc ───────────────────────────────────────
        ds = xr.open_dataset(nldas_path)
        nldas_et = ds['EVPsfc'].squeeze()  # shape: (lat, lon), mm/month
        nldas_et = nldas_et.rio.write_crs('EPSG:4326')

        # Clip to Iowa extent
        nldas_iowa = nldas_et.rio.clip(iowa_gdf.geometry, iowa_gdf.crs, drop=True)
        ds.close()

        # ── Load OpenET ──────────────────────────────────────────────────
        openet = rxr.open_rasterio(openet_path, masked=True).squeeze()

        # ── Align NLDAS to OpenET pixel grid ────────────────────────────
        # Reindex NLDAS to match OpenET lon/lat coordinates exactly
        nldas_aligned = nldas_iowa.interp(
            lon=openet.x.values,
            lat=openet.y.values,
            method='nearest'
        )

        # ── Compute delta ────────────────────────────────────────────────
        delta = openet.values - nldas_aligned.values  # mm/month

        # ── Save GeoTIFF ─────────────────────────────────────────────────
        delta_da = xr.DataArray(
            delta,
            coords={'y': openet.y, 'x': openet.x},
            dims=['y', 'x']
        )
        delta_da = delta_da.rio.write_crs('EPSG:4326')
        delta_da.rio.to_raster(out_path)

        # ── Summary stats ────────────────────────────────────────────────
        valid = delta[~np.isnan(delta)]
        records.append({
            'year': year, 'month': month, 'yyyymm': yyyymm,
            'openet_mean':  float(openet.values[~np.isnan(openet.values)].mean()),
            'nldas_mean':   float(nldas_aligned.values[~np.isnan(nldas_aligned.values)].mean()),
            'delta_mean':   float(valid.mean()),
            'delta_std':    float(valid.std()),
            'delta_min':    float(valid.min()),
            'delta_max':    float(valid.max()),
        })
        delta_grids[(year, month)] = delta_da

        print(f'  {yyyymm}: OpenET={records[-1]["openet_mean"]:6.1f}  NLDAS={records[-1]["nldas_mean"]:6.1f}  Δ={records[-1]["delta_mean"]:+6.1f} mm/month')

df = pd.DataFrame(records)
print(f'\nProcessed {len(df)} months, skipped {len(skipped)}')

## 4. Save Summary CSV

In [ ]:
csv_path = output_dir / 'ET_delta_monthly_summary_Iowa.csv'
df.to_csv(csv_path, index=False)
print(f'Saved: {csv_path}')
print(df.to_string(index=False))

## 5. Visualize — Time Series of Mean Delta

In [ ]:
dates = pd.to_datetime(df['yyyymm'], format='%Y%m')

fig, axes = plt.subplots(2, 1, figsize=(16, 10), sharex=True)

# Top: OpenET vs NLDAS Noah
axes[0].plot(dates, df['openet_mean'], 'o-', color='steelblue', label='OpenET (actual)', linewidth=1.5)
axes[0].plot(dates, df['nldas_mean'],  'o-', color='darkorange',  label='NLDAS Noah (modeled)', linewidth=1.5)
axes[0].set_ylabel('Mean ET (mm/month)', fontsize=11)
axes[0].set_title('OpenET vs NLDAS Noah ET — Iowa 2019–2023', fontsize=13, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Bottom: Delta
colors = ['tomato' if v > 0 else 'steelblue' for v in df['delta_mean']]
axes[1].bar(dates, df['delta_mean'], width=20, color=colors, alpha=0.8)
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].fill_between(dates,
    df['delta_mean'] - df['delta_std'],
    df['delta_mean'] + df['delta_std'],
    alpha=0.2, color='gray')
axes[1].set_ylabel('ΔET = OpenET − NLDAS (mm/month)', fontsize=11)
axes[1].set_title('Human ET Contribution (ΔET) — Positive = Excess ET (Irrigation Signal)', fontsize=13, fontweight='bold')
axes[1].grid(True, alpha=0.3)

for year in YEARS:
    axes[1].axvline(pd.Timestamp(f'{year}-01-01'), color='gray', linestyle=':', alpha=0.4)

plt.tight_layout()
plt.savefig(figures_dir / 'et_delta_timeseries.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Visualize — Seasonal Cycle of Delta

In [ ]:
colors_yr = {2019: '#1f77b4', 2020: '#ff7f0e', 2021: '#2ca02c', 2022: '#d62728', 2023: '#9467bd'}

fig, ax = plt.subplots(figsize=(12, 6))

for year in YEARS:
    yr = df[df['year'] == year].sort_values('month')
    if not yr.empty:
        ax.plot(yr['month'], yr['delta_mean'], 'o-',
                color=colors_yr[year], linewidth=2, markersize=6, label=str(year))

# 5-year mean seasonal cycle
monthly_mean = df.groupby('month')['delta_mean'].mean()
ax.plot(monthly_mean.index, monthly_mean.values, 'k--', linewidth=2.5, label='5-yr mean', zorder=5)

ax.axhline(0, color='black', linewidth=0.8)
ax.set_xticks(range(1, 13))
ax.set_xticklabels(['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'])
ax.set_xlabel('Month', fontsize=12)
ax.set_ylabel('ΔET (mm/month)', fontsize=12)
ax.set_title('Seasonal Cycle of Human ET Contribution — Iowa\nΔET = OpenET − NLDAS Noah', fontsize=13, fontweight='bold')
ax.legend(title='Year')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(figures_dir / 'et_delta_seasonal_cycle.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Visualize — Spatial Maps of Delta

In [ ]:
# Peak irrigation months: June, July, August for a sample year
sample_year = 2021
plot_months = [4, 6, 7, 8, 10]  # Apr, Jun, Jul, Aug, Oct
month_names = {4:'April', 6:'June', 7:'July', 8:'August', 10:'October'}

fig, axes = plt.subplots(1, len(plot_months), figsize=(20, 5))

# Symmetric colorbar centered on zero
all_vals = np.concatenate([
    delta_grids[(sample_year, m)].values.flatten()
    for m in plot_months
    if (sample_year, m) in delta_grids
])
vmax = np.nanpercentile(np.abs(all_vals), 95)
norm = mcolors.TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)

for ax, month in zip(axes, plot_months):
    if (sample_year, month) not in delta_grids:
        ax.set_visible(False)
        continue
    data = delta_grids[(sample_year, month)]
    im = ax.imshow(
        data.values,
        extent=[float(data.x.min()), float(data.x.max()),
                float(data.y.min()), float(data.y.max())],
        origin='upper', cmap='RdBu_r', norm=norm
    )
    iowa_gdf.boundary.plot(ax=ax, color='black', linewidth=0.8)
    mean_val = float(np.nanmean(data.values))
    ax.set_title(f'{month_names[month]} {sample_year}\nΔ={mean_val:+.1f} mm', fontsize=11, fontweight='bold')
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')

cbar = fig.colorbar(im, ax=axes, orientation='horizontal', fraction=0.04, pad=0.15)
cbar.set_label('ΔET = OpenET − NLDAS Noah (mm/month)  |  Red = Excess ET (irrigation signal)', fontsize=10)

plt.suptitle(f'Human ET Contribution — Iowa {sample_year}', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(figures_dir / f'et_delta_maps_{sample_year}.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Annual Mean Delta Heatmap

In [ ]:
pivot = df.pivot(index='year', columns='month', values='delta_mean')

vmax = np.nanpercentile(np.abs(pivot.values), 95)
norm = mcolors.TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)

fig, ax = plt.subplots(figsize=(14, 4))
im = ax.imshow(pivot.values, cmap='RdBu_r', norm=norm, aspect='auto')

ax.set_xticks(range(12))
ax.set_xticklabels(['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'])
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index)

for i in range(len(pivot.index)):
    for j in range(12):
        val = pivot.values[i, j]
        if not np.isnan(val):
            ax.text(j, i, f'{val:+.0f}', ha='center', va='center', fontsize=8,
                    color='white' if abs(val) > vmax * 0.5 else 'black')

cbar = plt.colorbar(im, ax=ax, shrink=0.8)
cbar.set_label('ΔET (mm/month)')
ax.set_title('ΔET Heatmap — Iowa 2019–2023  |  Red = Excess ET (irrigation signal)', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig(figures_dir / 'et_delta_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Summary

In [ ]:
print('=' * 65)
print('ET Harmonization Summary — Iowa 2019–2023')
print('=' * 65)
print(f'Months processed: {len(df)}')
print(f'Months skipped:   {len(skipped)}')
print()
print(f'Overall mean OpenET:    {df["openet_mean"].mean():.1f} mm/month')
print(f'Overall mean NLDAS Noah:{df["nldas_mean"].mean():.1f} mm/month')
print(f'Overall mean ΔET:       {df["delta_mean"].mean():+.1f} mm/month')
print()
print('Annual mean ΔET:')
for year, grp in df.groupby('year'):
    print(f'  {year}: {grp["delta_mean"].mean():+.1f} mm/month  (annual sum: {grp["delta_mean"].sum():+.0f} mm/yr)')
print()
print('Peak irrigation months (highest mean ΔET):')
top = df.groupby('month')['delta_mean'].mean().sort_values(ascending=False).head(4)
month_names_map = {1:'Jan',2:'Feb',3:'Mar',4:'Apr',5:'May',6:'Jun',
                   7:'Jul',8:'Aug',9:'Sep',10:'Oct',11:'Nov',12:'Dec'}
for m, v in top.items():
    print(f'  {month_names_map[m]}: {v:+.1f} mm/month')
print()
print(f'Output GeoTIFFs: {output_dir}')
print(f'Summary CSV:     {csv_path}')
print(f'Figures:         {figures_dir}')